In [1]:
import os
import zstandard  # pip install zstandard
from tqdm import tqdm
import random
import json
import langid
from typing import Generator, Optional, Set

files_processed_to_text = True




In [2]:
from datetime import datetime
def showTime():
    return str("["+datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')+" UTC]")

In [3]:
def zst_files_in_dir(directory):
    """List all .zst files in a directory."""
    files = []
    for filename in os.listdir(directory):
        if filename.endswith(".zst") and os.path.isfile(os.path.join(directory, filename)):
            files.append(filename)
    return files

In [4]:
def decompress_zst_to_text(
    input_file: str,
    vocab: Optional[Set[str]] = None,
    mode: str = "accuracy",
    ascii_threshold: float = 0.5
) -> Generator[str, None, None]:
    """
    Decompresses a .zst file containing JSONL (JSON lines) format, 
    and yields English texts filtered via language detection or ASCII checks.
    
    ### Parameters
    input_file (str):
        Path to the .zst file containing JSONL-formatted lines.
        
    vocab (Optional[Set[str]]):
        A vocabulary set to collect unique characters from valid English texts. Defaults to None.
        
    mode (str, default='accuracy'):
        - 'accuracy': Uses the `langid` library for precise English language detection.
        - 'speed': Uses an ASCII ratio check for faster filtering.
        
    ascii_threshold (float, default=0.5):
        Minimum ASCII character ratio (0.0-1.0) for mode='speed' to consider text as valid.
    
    ### Yields
    str:
        Filtered English text entries from the compressed file.
    """
    with open(input_file, "rb") as infile:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(infile) as reader:
            current_line = ""
            while True:
                chunk = reader.read(16384).decode("utf-8", errors="replace")  # Read in 16KB chunks
                if not chunk:
                    break
                current_line += chunk
                # Split into lines (handles partial lines)
                lines = current_line.split("\n")
                current_line = lines.pop() if lines else ""  # Save partial line for next iteration

                # Process each line
                for line in lines:
                    # Skip empty lines after stripping
                    stripped_line = line.strip()
                    if not stripped_line:
                        continue
                    
                    try:
                        data = json.loads(stripped_line)
                        text = data.get("text", "").strip()
                    except (json.JSONDecodeError, KeyError):
                        continue  # Skip invalid JSON
                        
                    except Exception as e:
                        print(f"JSON Error: {e} on line: {line[:50]}...")
                        continue

                    if mode == "accuracy":      
                        # Check language (English)
                        try:
                            detected_lang, _ = langid.classify(text)
                        except langid.langid.LanguageIdentificationError:
                            # Skip texts too short to identify
                            continue

                        if detected_lang != "en":
                            continue  # Non-English, skip

                    elif mode == "speed":
                        # ---- START FILTERING LOGIC ----
                        ascii_count = 0
                        total_chars = 0
                        
                        # Iterate through each character in text
                        for c in text:
                            code = ord(c)
                            if code <= 127:
                                ascii_count += 1
                            total_chars += 1

                        # Check filtering conditions
                        if total_chars == 0:
                            continue
                        if (ascii_count / total_chars) < ascii_threshold:
                            continue
                        # ---- END FILTERING LOGIC ----

                    # Update the vocabulary (only for English texts)
                    if vocab is not None:
                        vocab.update(set(text))

                    yield text.strip()


In [5]:
folder_path = "openwebtext2"
output_file = "output_v7_accuracy.txt"
vocab_file = "vocab_v7_accuracy.txt"


In [ ]:
# Gather files
files = zst_files_in_dir(folder_path)
total_files = len(files)
print(f"Total files: {total_files}")
print(files)
vocab = set()

In [ ]:
# Shuffle files randomly 
random.seed(42)  # Optional: Set seed for reproducibility
random.shuffle(files)  # Shuffle in-place
print(files)

In [8]:
# Process all files
if files_processed_to_text == False:
    with open(output_file, "w", encoding="utf-8") as outf:
        for filename in tqdm(files, total=len(files), desc="Processing Files"):
            print(f"{showTime()} Processing: {filename}")
            file_path = os.path.join(folder_path, filename)
            try:
                for text_line in decompress_zst_to_text(file_path, vocab, mode="accuracy"):
                    outf.write(text_line.strip())  # Write only the text line
            except Exception as e:
                print(f"Error processing {file_path}: {e}")

In [9]:
# Write vocabulary
if files_processed_to_text == False:
    with open(vocab_file, "w", encoding="utf-8") as vfile:
        for char in sorted(vocab):
            vfile.write(char + "\n")

In [ ]:
#load sequence
with open(output_file, "r", encoding="utf-8") as f:
    number_of_characters_to_read = 10_000_000
    text_sequence = f.read(number_of_characters_to_read)

len(text_sequence)

In [ ]:
# Karpathy minBPE repository
from minbpe import RegexTokenizer

tokenizer = RegexTokenizer()
tokenizer.train(text_sequence, vocab_size=16_384)

In [ ]:

tokenizer = RegexTokenizer()
tokenizer.train(text_sequence, vocab_size=16_384)

In [ ]:
vocab = tokenizer.vocab
vocab

In [ ]:
encoded_text = tokenizer.encode("Hello, world! I like apple juice - I drink it every day. Isn't that too much?")
print(encoded_text)

In [ ]:
decoded_text = tokenizer.decode(encoded_text)
print(decoded_text)

In [27]:
max_vocab_id = list(tokenizer.vocab.keys())[-1]
tokenizer.special_tokens = {
    "<|startoftext|>": max_vocab_id + 1,
    "<|separator|>": max_vocab_id + 2,
    "<|endoftext|>": max_vocab_id + 3,
    "<|unk|>": max_vocab_id + 4,
    "<|padding|>": max_vocab_id + 5
}

In [4]:
tokenizer_output_dir = "output_v7/tokenizer"
tokenizer_path = os.path.join(tokenizer_output_dir, "en_tokenizer")


In [ ]:
import os
if not os.path.exists(tokenizer_output_dir):
    os.makedirs(tokenizer_output_dir)


tokenizer.save(file_prefix=tokenizer_path)

Encoding the sequence of text

In [3]:
import os
from minbpe import RegexTokenizer
tokenizer = RegexTokenizer()
TOKENIZER_DIR    = os.path.join("output_v7", "tokenizer")
TOKENIZER_MODEL  = os.path.join(TOKENIZER_DIR, "en_tokenizer.model")
tokenizer.load(model_file=TOKENIZER_MODEL)
print(tokenizer.special_tokens)

{'<|startoftext|>': 16384, '<|separator|>': 16385, '<|endoftext|>': 16386, '<|unk|>': 16387, '<|padding|>': 16388}


In [1]:
import os
import logging
import gc
import numpy as np
from time import time
from joblib import Parallel, delayed
from minbpe import RegexTokenizer

# ── Logging Configuration ──────────────────────────────────────────────
logging.basicConfig(
    level=logging.DEBUG,  # switch to INFO or WARNING to reduce verbosity
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

def show_time():
    return f"[{time():.2f}]"

# ── Configuration ──────────────────────────────────────────────────────
FILE_PATH       = "output_v7_accuracy.txt"
TOKENIZER_MODEL = os.path.join("output_v7", "tokenizer", "en_tokenizer.model")
BATCH_SIZE      = 100_000_000      # bytes per read()
ENC_DIR         = os.path.join("output_v7", "encoded_data")
OUTPUT_FILENAME = "encoded_output_v7_accuracy.npy"
n_jobs          = min(max(os.cpu_count() // 2, 1), 8)

os.makedirs(ENC_DIR, exist_ok=True)
logger.info(f"Config loaded: FILE_PATH={FILE_PATH}, BATCH_SIZE={BATCH_SIZE}, "
            f"TOKENIZER_MODEL={TOKENIZER_MODEL}, ENC_DIR={ENC_DIR}, n_jobs={n_jobs}")

# ── Load Tokenizer & Special Tokens ────────────────────────────────────
logger.debug("Initializing tokenizer and loading special tokens...")
tokenizer = RegexTokenizer()
tokenizer.load(model_file=TOKENIZER_MODEL)
special_tokens = sorted(tokenizer.special_tokens.keys(), key=len, reverse=True)
logger.info(f"Loaded {len(special_tokens)} special tokens: {special_tokens}")

# ── Chunk Reader with Detailed Logging ─────────────────────────────────
def chunk_reader(fp, size):
    leftover = ""
    total_read = 0
    chunk_index = 0

    while True:
        logger.debug(f"{show_time()} Reading up to {size} bytes from file...")
        data = fp.read(size)
        read_len = len(data)
        total_read += read_len
        logger.info(f"{show_time()} Read chunk #{chunk_index} ({read_len} bytes, total read {total_read} bytes)")

        if not data:
            if leftover:
                logger.debug(f"{show_time()} Yielding final leftover ({len(leftover)} chars)")
                yield leftover
            logger.info(f"{show_time()} chunk_reader: EOF reached after {total_read} bytes")
            break

        text = leftover + data.decode("utf-8", errors="ignore")
        logger.debug(f"{show_time()} Decoded to text ({len(text)} chars incl. leftover)")

        # find split boundary
        last_ws = text.rstrip().rfind(" ")
        last_tok_end = -1
        for tok in special_tokens:
            idx = text.rfind(tok)
            if idx != -1:
                last_tok_end = max(last_tok_end, idx + len(tok))

        split_at = max(last_ws + 1, last_tok_end)
        if split_at <= 0:
            logger.warning(f"{show_time()} No safe split found in chunk #{chunk_index}; using full text")
            yield text
            leftover = ""
        else:
            safe = text[:split_at]
            leftover = text[split_at:]
            logger.info(f"{show_time()} chunk #{chunk_index} split at {split_at} → safe ({len(safe)} chars), leftover ({len(leftover)} chars)")
            yield safe

        chunk_index += 1

# ── Tokenize Worker with Context Logging ───────────────────────────────
chunks_dropped = 0

def tokenize_chunk(text_chunk, idx):
    global chunks_dropped
    logger.debug(f"{show_time()} Worker {os.getpid()} received chunk #{idx} ({len(text_chunk)} chars)")
    try:
        tokens = tokenizer.encode(text_chunk)
        logger.info(f"{show_time()} Worker {os.getpid()}: chunk #{idx} → {len(tokens)} tokens")
        return tokens
    except AssertionError:
        frags = [tok for tok in special_tokens if tok in text_chunk]
        logger.error(f"{show_time()} Dropping chunk #{idx}; partial special-token fragments: {frags}")
        chunks_dropped += 1
        return []

# ── Main Parallel Pipeline with Logging ─────────────────────────────────
def run_parallel_encoding():
    logger.info("Starting parallel encoding pipeline")

    temp_files   = []
    total_tokens = 0
    chunk_count  = 0
    batch_count  = 0

    with open(FILE_PATH, "rb") as f:
        reader = chunk_reader(f, BATCH_SIZE)

        while True:
            group = []
            for _ in range(n_jobs):
                try:
                    chunk = next(reader)
                    group.append(chunk)
                except StopIteration:
                    break

            if not group:
                logger.info("No more chunks to process; ending loop")
                break

            logger.info(f"{show_time()} Batch #{batch_count} | Dispatching {len(group)} chunks to {n_jobs} workers")
            results = Parallel(n_jobs=n_jobs, backend="loky", verbose=5)(
                delayed(tokenize_chunk)(chunk, chunk_count + i)
                for i, chunk in enumerate(group)
            )

            # collect and write
            batch_tokens = []
            for tokens in results:
                batch_tokens.extend(tokens)
                total_tokens += len(tokens)
                chunk_count += 1
                logger.debug(f"{show_time()} Collected {len(tokens)} tokens from chunk #{chunk_count}")

                del tokens
                gc.collect()

            tmp_path = os.path.join(ENC_DIR, f"batch_{batch_count:04d}.npy")
            np.save(tmp_path, np.array(batch_tokens, dtype=np.int64))
            temp_files.append(tmp_path)
            logger.info(f"{show_time()} Wrote batch file {tmp_path} with {len(batch_tokens)} tokens")

            del batch_tokens, group
            gc.collect()

            batch_count += 1

    # concatenate
    output_path = os.path.join(ENC_DIR, OUTPUT_FILENAME)
    logger.info(f"{show_time()} Concatenating {len(temp_files)} batch files into {output_path}")
    with open(output_path, "wb") as out_f:
        for tmp in temp_files:
            logger.debug(f"{show_time()} Reading temp file {tmp}")
            with open(tmp, "rb") as in_f:
                out_f.write(in_f.read())
            os.remove(tmp)
            logger.info(f"{show_time()} Appended and removed {tmp}")

    logger.info(f"{show_time()} Pipeline complete: {total_tokens} tokens, {chunks_dropped} chunks dropped")
    logger.info(f"Output saved to {output_path}")

if __name__ == "__main__":
    run_parallel_encoding()


04:18:10 [INFO] Config loaded: FILE_PATH=output_v7_accuracy.txt, BATCH_SIZE=100000000, TOKENIZER_MODEL=output_v7\tokenizer\en_tokenizer.model, ENC_DIR=output_v7\encoded_data, n_jobs=8
04:18:10 [DEBUG] Initializing tokenizer and loading special tokens...
04:18:10 [INFO] Loaded 5 special tokens: ['<|startoftext|>', '<|separator|>', '<|endoftext|>', '<|padding|>', '<|unk|>']
04:18:10 [INFO] Starting parallel encoding pipeline
04:18:10 [DEBUG] [1746325090.60] Reading up to 100000000 bytes from file...
04:18:10 [INFO] [1746325090.62] Read chunk #0 (100000000 bytes, total read 100000000 bytes)
04:18:10 [DEBUG] [1746325090.68] Decoded to text (99069447 chars incl. leftover)
04:18:10 [INFO] [1746325090.98] chunk #0 split at 99069444 → safe (99069444 chars), leftover (3 chars)
04:18:10 [DEBUG] [1746325090.98] Reading up to 100000000 bytes from file...
04:18:11 [INFO] [1746325091.00] Read chunk #1 (100000000 bytes, total read 200000000 bytes)
04:18:11 [DEBUG] [1746325091.17] Decoded to text (990